## TOG Attacks on YOLOv3

In [ ]:
from __future__ import annotations

from collections.abc import Callable, Iterator, Sequence
from pathlib import Path
from typing import Literal, Optional

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from scipy.special import softmax
from torchvision.ops import nms

ROOT = Path('.').resolve()


Defines some helpful utility functions which are being used for this project.


In [ ]:
COCO_CLASSES = [
    "person", "bicycle", "car", "motorcycle", "airplane", "bus", "train", "truck", "boat",
    "traffic light", "fire hydrant", "stop sign", "parking meter", "bench", "bird", "cat",
    "dog", "horse", "sheep", "cow", "elephant", "bear", "zebra", "giraffe", "backpack",
    "umbrella", "handbag", "tie", "suitcase", "frisbee", "skis", "snowboard", "sports ball",
    "kite", "baseball bat", "baseball glove", "skateboard", "surfboard", "tennis racket",
    "bottle", "wine glass", "cup", "fork", "knife", "spoon", "bowl", "banana", "apple",
    "sandwich", "orange", "broccoli", "carrot", "hot dog", "pizza", "donut", "cake", "chair",
    "couch", "potted plant", "bed", "dining table", "toilet", "tv", "laptop", "mouse",
    "remote", "keyboard", "cell phone", "microwave", "oven", "toaster", "sink", "refrigerator",
    "book", "clock", "vase", "scissors", "teddy bear", "hair drier", "toothbrush",
]

MislabelingMode = Literal["most_likely", "least_likely"]


def letterbox_image(image: Image.Image, size: tuple[int, int] = (416, 416)) -> tuple[np.ndarray, tuple[int, int, int, int, float]]:
    """Resize an image with aspect-ratio-preserving padding and return NHWC float data."""
    source = image.copy()
    source_width, source_height = source.size
    target_width, target_height = size
    scale = min(target_width / source_width, target_height / source_height)
    resized_width = int(source_width * scale)
    resized_height = int(source_height * scale)

    source = source.resize((resized_width, resized_height), Image.Resampling.BICUBIC)
    padded = Image.new("RGB", size, (0, 0, 0))
    left = (target_width - resized_width) // 2
    top = (target_height - resized_height) // 2
    padded.paste(source, (left, top))

    image_array = np.asarray(padded, dtype=np.float32)[None, ...] / 255.0
    metadata = (left, top, left + resized_width, top + resized_height, scale)
    return image_array, metadata


def visualize_detections(detection_sets: dict) -> None:
    """Plot one or more images with detector outputs in the repository detection format."""
    if not detection_sets:
        raise ValueError("detection_sets must contain at least one item.")

    num_colors = max(21, max(len(item[3]) for item in detection_sets.values()))
    colors = plt.cm.hsv(np.linspace(0, 1, num_colors)).tolist()
    plt.figure(figsize=(3 * len(detection_sets), 3))

    for plot_index, (title, values) in enumerate(detection_sets.items(), start=1):
        image, detections, model_image_size, class_names = values
        image = image[0] if image.ndim == 4 else image
        axis = plt.subplot(1, len(detection_sets), plot_index)
        axis.set_title(title)
        axis.imshow(image)

        for detection in detections:
            class_id = int(detection[0])
            xmin = max(int(detection[-4] * image.shape[1] / model_image_size[1]), 0)
            ymin = max(int(detection[-3] * image.shape[0] / model_image_size[0]), 0)
            xmax = min(int(detection[-2] * image.shape[1] / model_image_size[1]), image.shape[1])
            ymax = min(int(detection[-1] * image.shape[0] / model_image_size[0]), image.shape[0])
            color = colors[class_id % num_colors]
            label = f"{class_names[class_id]}: {detection[1]:.2f}"
            axis.add_patch(
                plt.Rectangle((xmin, ymin), xmax - xmin, ymax - ymin, color=color, fill=False, linewidth=2)
            )
            axis.text(xmin, ymin, label, size="small", color="black", bbox={"facecolor": color, "alpha": 1.0})
        axis.axis("off")

    plt.tight_layout()
    plt.show()


def generate_attack_targets(
    detections: np.ndarray,
    mode: MislabelingMode,
    confidence_threshold: float,
    source_class_id: int | None = None,
) -> np.ndarray:
    """Replace detected class IDs with TOG most- or least-likely target classes."""
    if mode not in ("most_likely", "least_likely"):
        raise ValueError("mode must be 'most_likely' or 'least_likely'.")
    if not 0.0 <= confidence_threshold <= 1.0:
        raise ValueError("confidence_threshold must be between 0 and 1.")

    detections = np.asarray(detections, dtype=np.float32)
    if detections.ndim != 2 or detections.shape[0] == 0:
        raise ValueError("Mislabeling requires at least one valid detection.")

    targets = detections.copy()
    class_logits = targets[:, 2:-4].copy()
    if class_logits.shape[1] == 0:
        raise ValueError("Detections do not contain class logits.")

    if mode == "least_likely":
        target_class_ids = np.argmin(class_logits, axis=1)
    else:
        confident_classes = softmax(class_logits, axis=1) > confidence_threshold
        class_logits[confident_classes] = -np.inf
        target_class_ids = np.argmax(class_logits, axis=1)

    if source_class_id is not None:
        source_mask = targets[:, 0].astype(np.int64) == source_class_id
        if not source_mask.any():
            raise ValueError(f"No detections found for source class {source_class_id}.")
        target_class_ids = np.where(source_mask, target_class_ids, targets[:, 0].astype(np.int64))

    targets[:, 0] = target_class_ids.astype(np.float32)
    targets[:, 1] = 1.0
    return targets


### YOLOv3 detector

The code cell below implements the YOLOv3 architecture using Pytorch. The architectures and layers are implemented following standard YOLOv3. For this project, we load the open source trained weights of YOLOv3.

| Index | Meaning | Used as |
|------:|---------|---------|
| `0` | \(t_x\) | center x (logit / raw) |
| `1` | \(t_y\) | center y |
| `2` | \(t_w\) | width |
| `3` | \(t_h\) | height |
| `4` | objectness | “is there an object?” |
| `5 … 84` | class logits | one logit per COCO class |

In [ ]:
DEFAULT_ANCHORS = np.asarray(
    [[10, 13], [16, 30], [33, 23], [30, 61], [62, 45], [59, 119], [116, 90], [156, 198], [373, 326]],
    dtype=np.float32,
)
ANCHOR_MASKS = [[6, 7, 8], [3, 4, 5], [0, 1, 2]]


def to_nchw(image: np.ndarray | torch.Tensor, device: torch.device) -> torch.Tensor:
    """Convert NHWC or NCHW image data to a validated NCHW float tensor."""
    tensor = torch.from_numpy(image.astype(np.float32, copy=False)) if isinstance(image, np.ndarray) else image.float()
    if tensor.ndim == 3:
        tensor = tensor.unsqueeze(0)
    if tensor.ndim != 4:
        raise ValueError("image must have three or four dimensions.")
    if tensor.shape[-1] == 3:
        tensor = tensor.permute(0, 3, 1, 2).contiguous()
    elif tensor.shape[1] != 3:
        raise ValueError("image must contain exactly three channels.")
    if not torch.isfinite(tensor).all():
        raise ValueError("image contains non-finite values.")
    return tensor.to(device)


class ConvBNLeaky(nn.Module):
    """Apply a Darknet convolution, batch normalization, and leaky ReLU."""

    def __init__(self, input_channels: int, output_channels: int, kernel_size: int = 3, stride: int = 1):
        """Initialize one Darknet convolution block."""
        super().__init__()
        padding = (kernel_size - 1) // 2 if stride == 1 else 0
        self.asymmetric_padding = nn.ZeroPad2d((1, 0, 1, 0)) if stride == 2 else None
        self.convolution = nn.Conv2d(
            input_channels,
            output_channels,
            kernel_size,
            stride=stride,
            padding=padding,
            bias=False,
        )
        self.batch_norm = nn.BatchNorm2d(output_channels, momentum=0.03, eps=1e-4)
        self.activation = nn.LeakyReLU(0.1, inplace=True)

    def forward(self, inputs: torch.Tensor) -> torch.Tensor:
        """Run the convolution block."""
        if self.asymmetric_padding is not None:
            inputs = self.asymmetric_padding(inputs)
        return self.activation(self.batch_norm(self.convolution(inputs)))


class ResidualBlock(nn.Module):
    """Apply one Darknet-53 residual block."""

    def __init__(self, channels: int):
        """Initialize a bottleneck residual block."""
        super().__init__()
        self.reduce = ConvBNLeaky(channels, channels // 2, kernel_size=1)
        self.expand = ConvBNLeaky(channels // 2, channels, kernel_size=3)

    def forward(self, inputs: torch.Tensor) -> torch.Tensor:
        """Add the residual transformation to its input."""
        return inputs + self.expand(self.reduce(inputs))


class ResidualStage(nn.Module):
    """Downsample once and apply a sequence of residual blocks."""

    def __init__(self, input_channels: int, output_channels: int, num_blocks: int):
        """Initialize one Darknet-53 stage."""
        super().__init__()
        self.downsample = ConvBNLeaky(input_channels, output_channels, kernel_size=3, stride=2)
        self.blocks = nn.Sequential(*[ResidualBlock(output_channels) for _ in range(num_blocks)])

    def forward(self, inputs: torch.Tensor) -> torch.Tensor:
        """Run the downsampling and residual blocks."""
        return self.blocks(self.downsample(inputs))


class Darknet53(nn.Module):
    """Produce Darknet-53 feature maps at strides 8, 16, and 32."""

    def __init__(self):
        """Initialize the Darknet-53 backbone."""
        super().__init__()
        self.stem = ConvBNLeaky(3, 32)
        self.stage1 = ResidualStage(32, 64, 1)
        self.stage2 = ResidualStage(64, 128, 2)
        self.stage3 = ResidualStage(128, 256, 8)
        self.stage4 = ResidualStage(256, 512, 8)
        self.stage5 = ResidualStage(512, 1024, 4)

    def forward(self, inputs: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        """Return small-, medium-, and large-scale backbone features."""
        inputs = self.stem(inputs)
        inputs = self.stage1(inputs)
        inputs = self.stage2(inputs)
        small_features = self.stage3(inputs)
        medium_features = self.stage4(small_features)
        large_features = self.stage5(medium_features)
        return small_features, medium_features, large_features


class YOLOHead(nn.Module):
    """Build one YOLOv3 multi-convolution prediction head."""

    def __init__(self, input_channels: int, hidden_channels: int, output_channels: int):
        """Initialize a YOLOv3 detection head."""
        super().__init__()
        self.features = nn.Sequential(
            ConvBNLeaky(input_channels, hidden_channels, kernel_size=1),
            ConvBNLeaky(hidden_channels, hidden_channels * 2, kernel_size=3),
            ConvBNLeaky(hidden_channels * 2, hidden_channels, kernel_size=1),
            ConvBNLeaky(hidden_channels, hidden_channels * 2, kernel_size=3),
            ConvBNLeaky(hidden_channels * 2, hidden_channels, kernel_size=1),
        )
        self.prediction = nn.Sequential(
            ConvBNLeaky(hidden_channels, hidden_channels * 2, kernel_size=3),
            nn.Conv2d(hidden_channels * 2, output_channels, 1, bias=True),
        )

    def forward(self, inputs: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        """Return intermediate features and raw predictions."""
        features = self.features(inputs)
        return features, self.prediction(features)


class YOLOv3(nn.Module):
    """Return raw YOLOv3 predictions for three object scales."""

    def __init__(self, num_classes: int = 80, anchors_per_scale: int = 3):
        """Initialize the YOLOv3 backbone and prediction heads."""
        super().__init__()
        self.num_classes = num_classes
        self.anchors_per_scale = anchors_per_scale
        output_channels = anchors_per_scale * (5 + num_classes)

        self.backbone = Darknet53()
        self.large_head = YOLOHead(1024, 512, output_channels)
        self.large_upsample = nn.Sequential(
            ConvBNLeaky(512, 256, kernel_size=1),
            nn.Upsample(scale_factor=2, mode="nearest"),
        )
        self.medium_head = YOLOHead(768, 256, output_channels)
        self.medium_upsample = nn.Sequential(
            ConvBNLeaky(256, 128, kernel_size=1),
            nn.Upsample(scale_factor=2, mode="nearest"),
        )
        self.small_head = YOLOHead(384, 128, output_channels)

    def _reshape_prediction(self, prediction: torch.Tensor) -> torch.Tensor:
        """Reshape a convolution output to batch-anchor-grid-channel layout."""
        batch_size, _, grid_height, grid_width = prediction.shape
        prediction = prediction.view(
            batch_size,
            self.anchors_per_scale,
            5 + self.num_classes,
            grid_height,
            grid_width,
        )
        return prediction.permute(0, 1, 3, 4, 2).contiguous()

    def forward(self, inputs: torch.Tensor) -> list[torch.Tensor]:
        """Return large-, medium-, and small-object raw predictions."""
        small_features, medium_features, large_features = self.backbone(inputs)
        large_head_features, large_prediction = self.large_head(large_features)
        medium_input = torch.cat([self.large_upsample(large_head_features), medium_features], dim=1)
        medium_head_features, medium_prediction = self.medium_head(medium_input)
        small_input = torch.cat([self.medium_upsample(medium_head_features), small_features], dim=1)
        _, small_prediction = self.small_head(small_input)
        return [
            self._reshape_prediction(large_prediction),
            self._reshape_prediction(medium_prediction),
            self._reshape_prediction(small_prediction),
        ]


def _conv_blocks(module: nn.Module) -> Iterator[ConvBNLeaky]:
    """Yield Darknet convolution blocks in module registration order."""
    for child in module.modules():
        if isinstance(child, ConvBNLeaky):
            yield child


def _ordered_weight_layers(model: YOLOv3) -> Iterator[tuple[nn.Conv2d, nn.BatchNorm2d | None]]:
    """Yield convolution layers in the order used by Darknet weight files."""
    for block in _conv_blocks(model.backbone):
        yield block.convolution, block.batch_norm

    for head, upsample in (
        (model.large_head, model.large_upsample),
        (model.medium_head, model.medium_upsample),
        (model.small_head, None),
    ):
        for block in _conv_blocks(head.features):
            yield block.convolution, block.batch_norm
        prediction_block = head.prediction[0]
        yield prediction_block.convolution, prediction_block.batch_norm
        yield head.prediction[1], None
        if upsample is not None:
            for block in _conv_blocks(upsample):
                yield block.convolution, block.batch_norm


def load_darknet_weights(model: YOLOv3, weights_path: str | Path) -> None:
    """Load an official Darknet YOLOv3 binary weight file exactly."""
    with Path(weights_path).open("rb") as file:
        header = np.fromfile(file, dtype=np.int32, count=5)
        weights = np.fromfile(file, dtype=np.float32)
    if header.size != 5:
        raise RuntimeError("Invalid Darknet weight header.")

    pointer = 0

    def take(parameter: torch.Tensor) -> torch.Tensor:
        """Read and reshape the next parameter-sized slice from the weight array."""
        nonlocal pointer
        size = parameter.numel()
        if pointer + size > weights.size:
            raise RuntimeError("Darknet weight file ended unexpectedly.")
        values = torch.from_numpy(weights[pointer:pointer + size]).to(parameter.device, parameter.dtype)
        pointer += size
        return values.view_as(parameter)

    with torch.no_grad():
        for convolution, batch_norm in _ordered_weight_layers(model):
            if batch_norm is not None:
                batch_norm.bias.copy_(take(batch_norm.bias))
                batch_norm.weight.copy_(take(batch_norm.weight))
                batch_norm.running_mean.copy_(take(batch_norm.running_mean))
                batch_norm.running_var.copy_(take(batch_norm.running_var))
            else:
                if convolution.bias is None:
                    raise RuntimeError("Expected a bias tensor for a prediction convolution.")
                convolution.bias.copy_(take(convolution.bias))
            convolution.weight.copy_(take(convolution.weight))

    if pointer != weights.size:
        raise RuntimeError(f"Darknet weight mismatch: consumed {pointer}/{weights.size} values.")


class YOLOv3Detector:
    """Load YOLOv3 weights and expose detector-only inference operations."""

    def __init__(
        self,
        weights: str | Path = "weights/yolov3.weights",
        model_image_size: tuple[int, int] = (416, 416),
        confidence_threshold: float = 0.20,
        num_classes: Optional[int] = None,
        class_names: Optional[Sequence[str]] = None,
        device: Optional[str] = None,
        anchors: Optional[np.ndarray] = None,
    ):
        """Initialize the network, detector settings, and Darknet weights."""
        self.model_image_size = tuple(model_image_size)
        self.confidence_threshold = confidence_threshold
        self.class_names = list(class_names) if class_names is not None else list(COCO_CLASSES)
        self.num_classes = num_classes if num_classes is not None else len(self.class_names)
        if len(self.class_names) != self.num_classes:
            raise ValueError("class_names length must equal num_classes.")
        self.anchors = np.asarray(anchors, dtype=np.float32) if anchors is not None else DEFAULT_ANCHORS.copy()
        if self.anchors.shape != (9, 2):
            raise ValueError("anchors must have shape (9, 2).")
        if not 0.0 <= self.confidence_threshold <= 1.0:
            raise ValueError("confidence_threshold must be between 0 and 1.")

        self.device = torch.device(device or ("cuda" if torch.cuda.is_available() else "cpu"))
        self.network = YOLOv3(num_classes=self.num_classes).to(self.device)
        load_darknet_weights(self.network, weights)
        self.network.eval()

    def _decode_layer(
        self,
        prediction: torch.Tensor,
        anchors: np.ndarray,
    ) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        """Decode one raw YOLO layer into boxes, class scores, and class logits."""
        batch_size, num_anchors, grid_height, grid_width, _ = prediction.shape
        if batch_size != 1:
            raise ValueError("detect currently supports a batch size of one.")

        anchor_tensor = torch.as_tensor(anchors, dtype=prediction.dtype, device=prediction.device).view(
            1, num_anchors, 1, 1, 2
        )
        grid_y, grid_x = torch.meshgrid(
            torch.arange(grid_height, device=prediction.device),
            torch.arange(grid_width, device=prediction.device),
            indexing="ij",
        )
        grid = torch.stack((grid_x, grid_y), dim=-1).view(1, 1, grid_height, grid_width, 2).to(prediction.dtype)
        raw = prediction[0]
        grid_size = torch.tensor([grid_width, grid_height], device=prediction.device, dtype=prediction.dtype)
        input_size = torch.tensor(
            [self.model_image_size[1], self.model_image_size[0]],
            device=prediction.device,
            dtype=prediction.dtype,
        )
        box_xy = (torch.sigmoid(raw[..., :2]) + grid[0]) / grid_size
        box_wh = torch.exp(raw[..., 2:4].clamp(max=10)) * anchor_tensor[0] / input_size
        confidence = torch.sigmoid(raw[..., 4:5])
        class_logits = raw[..., 5:]
        class_scores = confidence * torch.sigmoid(class_logits)

        box_min = box_xy - box_wh / 2
        box_max = box_xy + box_wh / 2
        boxes = torch.cat([box_min, box_max], dim=-1) * torch.tensor(
            [self.model_image_size[1], self.model_image_size[0], self.model_image_size[1], self.model_image_size[0]],
            device=prediction.device,
            dtype=prediction.dtype,
        )
        return (
            boxes.reshape(-1, 4),
            class_scores.reshape(-1, self.num_classes),
            class_logits.reshape(-1, self.num_classes),
        )

    @torch.no_grad()
    def detect(
        self,
        image: np.ndarray,
        iou_threshold: float = 0.45,
        confidence_threshold: float | None = None,
        max_detections_per_class: int = 400,
    ) -> np.ndarray:
        """Detect objects and return class, score, logits, and xyxy box columns."""
        threshold = self.confidence_threshold if confidence_threshold is None else confidence_threshold
        if not 0.0 <= threshold <= 1.0:
            raise ValueError("confidence_threshold must be between 0 and 1.")
        if not 0.0 <= iou_threshold <= 1.0:
            raise ValueError("iou_threshold must be between 0 and 1.")
        if max_detections_per_class <= 0:
            raise ValueError("max_detections_per_class must be positive.")

        predictions = self.network(to_nchw(image, self.device))
        decoded = [
            self._decode_layer(prediction, self.anchors[mask])
            for prediction, mask in zip(predictions, ANCHOR_MASKS)
        ]
        boxes = torch.cat([item[0] for item in decoded], dim=0)
        scores = torch.cat([item[1] for item in decoded], dim=0)
        logits = torch.cat([item[2] for item in decoded], dim=0)

        rows = []
        for class_id in range(self.num_classes):
            class_scores = scores[:, class_id]
            candidate_mask = class_scores >= threshold
            if not candidate_mask.any():
                continue
            candidate_boxes = boxes[candidate_mask]
            candidate_scores = class_scores[candidate_mask]
            candidate_logits = logits[candidate_mask]
            kept_indices = nms(candidate_boxes, candidate_scores, iou_threshold)[:max_detections_per_class]
            for index in kept_indices:
                rows.append(
                    torch.cat(
                        [
                            torch.tensor([float(class_id), float(candidate_scores[index])], device=self.device),
                            candidate_logits[index],
                            candidate_boxes[index],
                        ]
                    ).cpu().numpy()
                )

        if not rows:
            return np.empty((0, 2 + self.num_classes + 4), dtype=np.float32)
        detections = np.stack(rows).astype(np.float32)
        return detections[np.argsort(detections[:, 1])[::-1]]


### Extract TOG losses


The code cell below implement a wrapper to extract the required losses from YOLOv3 for TOG attacks, and compute the gradient based on each type of attacks

In [ ]:
def box_iou_xywh(boxes1: torch.Tensor, boxes2: torch.Tensor) -> torch.Tensor:
    """Compute pairwise IoU between center-format xywh boxes."""
    boxes1 = boxes1.unsqueeze(-2)
    boxes2 = boxes2.unsqueeze(0)
    boxes1_min = boxes1[..., :2] - boxes1[..., 2:] / 2
    boxes1_max = boxes1[..., :2] + boxes1[..., 2:] / 2
    boxes2_min = boxes2[..., :2] - boxes2[..., 2:] / 2
    boxes2_max = boxes2[..., :2] + boxes2[..., 2:] / 2
    intersection_min = torch.maximum(boxes1_min, boxes2_min)
    intersection_max = torch.minimum(boxes1_max, boxes2_max)
    intersection_size = (intersection_max - intersection_min).clamp(min=0)
    intersection = intersection_size[..., 0] * intersection_size[..., 1]
    area1 = boxes1[..., 2] * boxes1[..., 3]
    area2 = boxes2[..., 2] * boxes2[..., 3]
    return intersection / (area1 + area2 - intersection + 1e-6)


def encode_yolo_targets(
    boxes: np.ndarray,
    input_shape: tuple[int, int],
    anchors: np.ndarray,
    num_classes: int,
) -> list[np.ndarray]:
    """Encode absolute xyxy boxes and class IDs into three YOLO target tensors."""
    boxes = np.asarray(boxes, dtype=np.float32)
    if boxes.ndim != 3 or boxes.shape[-1] != 5:
        raise ValueError("boxes must have shape (batch, num_boxes, 5).")
    if boxes.size and ((boxes[..., 4] < 0).any() or (boxes[..., 4] >= num_classes).any()):
        raise ValueError("Every class ID must be within the detector class range.")

    input_shape_array = np.asarray(input_shape, dtype=np.int32)
    normalized_boxes = boxes.copy()
    box_centers = (normalized_boxes[..., :2] + normalized_boxes[..., 2:4]) / 2
    box_sizes = normalized_boxes[..., 2:4] - normalized_boxes[..., :2]
    normalized_boxes[..., :2] = box_centers / input_shape_array[::-1]
    normalized_boxes[..., 2:4] = box_sizes / input_shape_array[::-1]

    grid_shapes = [input_shape_array // stride for stride in (32, 16, 8)]
    targets = [
        np.zeros(
            (boxes.shape[0], grid_height, grid_width, len(mask), 5 + num_classes),
            dtype=np.float32,
        )
        for (grid_height, grid_width), mask in zip(grid_shapes, ANCHOR_MASKS)
    ]

    anchor_boxes = anchors[None, ...]
    anchor_min = -anchor_boxes / 2
    anchor_max = anchor_boxes / 2
    valid_mask = (box_sizes[..., 0] > 0) & (box_sizes[..., 1] > 0)

    for batch_index in range(boxes.shape[0]):
        valid_indices = np.flatnonzero(valid_mask[batch_index])
        if valid_indices.size == 0:
            continue
        valid_sizes = box_sizes[batch_index, valid_indices, None, :]
        box_min = -valid_sizes / 2
        box_max = valid_sizes / 2
        intersection_size = np.maximum(np.minimum(box_max, anchor_max) - np.maximum(box_min, anchor_min), 0)
        intersection = intersection_size[..., 0] * intersection_size[..., 1]
        box_area = valid_sizes[..., 0] * valid_sizes[..., 1]
        anchor_area = anchor_boxes[..., 0] * anchor_boxes[..., 1]
        best_anchors = np.argmax(intersection / (box_area + anchor_area - intersection), axis=-1)

        for valid_position, anchor_index in enumerate(best_anchors):
            box_index = valid_indices[valid_position]
            for layer_index, anchor_mask in enumerate(ANCHOR_MASKS):
                if anchor_index not in anchor_mask:
                    continue
                grid_height, grid_width = grid_shapes[layer_index]
                grid_x = int(np.clip(np.floor(normalized_boxes[batch_index, box_index, 0] * grid_width), 0, grid_width - 1))
                grid_y = int(np.clip(np.floor(normalized_boxes[batch_index, box_index, 1] * grid_height), 0, grid_height - 1))
                mask_index = anchor_mask.index(int(anchor_index))
                class_id = int(normalized_boxes[batch_index, box_index, 4])
                targets[layer_index][batch_index, grid_y, grid_x, mask_index, :4] = normalized_boxes[
                    batch_index, box_index, :4
                ]
                targets[layer_index][batch_index, grid_y, grid_x, mask_index, 4] = 1.0
                targets[layer_index][batch_index, grid_y, grid_x, mask_index, 5 + class_id] = 1.0
    return targets


class YOLOv3TOGModel:
    """Wrap a detector with YOLOv3 loss and gradient operations for TOG."""

    def __init__(self, detector: YOLOv3Detector):
        """Store the detector used for inference and differentiable gradients."""
        self.detector = detector
        self.device = detector.device
        self.model_image_size = detector.model_image_size
        self.confidence_threshold = detector.confidence_threshold
        self.num_classes = detector.num_classes
        self.anchors = detector.anchors

    def detect(self, image: np.ndarray, **kwargs) -> np.ndarray:
        """Delegate inference to the separate object detector."""
        return self.detector.detect(image, **kwargs)

    def _decode_for_loss(
        self,
        prediction: torch.Tensor,
        anchors: np.ndarray,
    ) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        """Decode grid and normalized boxes needed by the YOLO training loss."""
        _, num_anchors, grid_height, grid_width, _ = prediction.shape
        anchor_tensor = torch.as_tensor(anchors, dtype=prediction.dtype, device=prediction.device).view(
            1, num_anchors, 1, 1, 2
        )
        grid_y, grid_x = torch.meshgrid(
            torch.arange(grid_height, device=prediction.device),
            torch.arange(grid_width, device=prediction.device),
            indexing="ij",
        )
        grid = torch.stack((grid_x, grid_y), dim=-1).view(1, 1, grid_height, grid_width, 2).to(prediction.dtype)
        grid_size = torch.tensor([grid_width, grid_height], device=prediction.device, dtype=prediction.dtype)
        input_size = torch.tensor(
            [self.model_image_size[1], self.model_image_size[0]],
            device=prediction.device,
            dtype=prediction.dtype,
        )
        box_xy = (torch.sigmoid(prediction[..., :2]) + grid) / grid_size
        box_wh = torch.exp(prediction[..., 2:4].clamp(max=10)) * anchor_tensor / input_size
        return grid, box_xy, box_wh

    def _objectness_loss(self, predictions: list[torch.Tensor], targets: list[torch.Tensor]) -> torch.Tensor:
        """Compute summed objectness binary cross-entropy across all YOLO scales."""
        loss = predictions[0].new_zeros(())
        for prediction, target in zip(predictions, targets):
            target = target.permute(0, 3, 1, 2, 4).contiguous()
            loss += F.binary_cross_entropy_with_logits(prediction[..., 4:5], target[..., 4:5], reduction="sum")
        return loss

    def _full_yolo_loss(self, predictions: list[torch.Tensor], targets: list[torch.Tensor]) -> torch.Tensor:
        """Compute the YOLOv3 box, objectness, and class loss used by TOG."""
        batch_size = float(predictions[0].shape[0])
        total_loss = predictions[0].new_zeros(())

        for prediction, target, anchor_mask in zip(predictions, targets, ANCHOR_MASKS):
            target = target.permute(0, 3, 1, 2, 4).contiguous()
            object_mask = target[..., 4:5]
            true_classes = target[..., 5:]
            grid, predicted_xy, predicted_wh = self._decode_for_loss(prediction, self.anchors[anchor_mask])
            predicted_boxes = torch.cat([predicted_xy, predicted_wh], dim=-1)
            grid_height, grid_width = prediction.shape[2:4]

            raw_true_xy = target[..., :2] * torch.tensor(
                [grid_width, grid_height], device=prediction.device, dtype=prediction.dtype
            ) - grid
            anchor_tensor = torch.as_tensor(
                self.anchors[anchor_mask], dtype=prediction.dtype, device=prediction.device
            ).view(1, len(anchor_mask), 1, 1, 2)
            raw_true_wh = torch.log(
                target[..., 2:4]
                * torch.tensor(
                    [self.model_image_size[1], self.model_image_size[0]],
                    device=prediction.device,
                    dtype=prediction.dtype,
                )
                / anchor_tensor
                + 1e-16
            )
            raw_true_wh = torch.where(object_mask.bool(), raw_true_wh, torch.zeros_like(raw_true_wh))
            box_loss_scale = 2.0 - target[..., 2:3] * target[..., 3:4]

            ignore_mask = torch.ones_like(object_mask)
            for batch_index in range(prediction.shape[0]):
                true_boxes = target[batch_index, ..., :4][object_mask[batch_index, ..., 0] > 0.5]
                if true_boxes.numel() == 0:
                    continue
                best_iou = box_iou_xywh(predicted_boxes[batch_index], true_boxes).max(dim=-1).values
                ignore_mask[batch_index, ..., 0] = (best_iou < 0.45).to(prediction.dtype)

            xy_loss = object_mask * box_loss_scale * F.binary_cross_entropy_with_logits(
                prediction[..., :2], raw_true_xy, reduction="none"
            )
            wh_loss = object_mask * box_loss_scale * 0.5 * (raw_true_wh - prediction[..., 2:4]) ** 2
            confidence_loss = object_mask * F.binary_cross_entropy_with_logits(
                prediction[..., 4:5], object_mask, reduction="none"
            ) + (1 - object_mask) * F.binary_cross_entropy_with_logits(
                prediction[..., 4:5], object_mask, reduction="none"
            ) * ignore_mask
            class_loss = object_mask * F.binary_cross_entropy_with_logits(
                prediction[..., 5:], true_classes, reduction="none"
            )
            total_loss += # TODO: Implement the total loss here using the loss components
        return total_loss

    def _targets_from_detections(self, detections: np.ndarray | None) -> list[torch.Tensor]:
        """Convert detector rows into YOLO target tensors on the detector device."""
        if detections is None or np.asarray(detections).size == 0:
            boxes = np.empty((1, 0, 5), dtype=np.float32)
        else:
            detections = np.asarray(detections, dtype=np.float32)
            if detections.ndim != 2 or detections.shape[1] < 6:
                raise ValueError("detections must be a two-dimensional detector output array.")
            boxes = detections[:, [-4, -3, -2, -1, 0]][None, ...]
        encoded = encode_yolo_targets(boxes, self.model_image_size, self.anchors, self.num_classes)
        return [torch.from_numpy(target).to(self.device) for target in encoded]

    def _image_gradient(
        self,
        image: np.ndarray,
        loss_function: Callable[[list[torch.Tensor]], torch.Tensor],
    ) -> np.ndarray:
        """Differentiate a supplied loss with respect to an NHWC input image."""
        input_tensor = to_nchw(image, self.device).detach().requires_grad_(True)
        self.detector.network.zero_grad(set_to_none=True)
        loss = loss_function(self.detector.network(input_tensor))
        loss.backward()
        if input_tensor.grad is None:
            raise RuntimeError("The attack loss did not produce an input gradient.")
        return input_tensor.grad.detach().permute(0, 2, 3, 1).cpu().numpy()

    def compute_object_vanishing_gradient(self, image: np.ndarray) -> np.ndarray:
        """Compute the gradient that minimizes objectness for every prediction cell."""
        targets = self._targets_from_detections(None)
        return self._image_gradient(image, lambda predictions: self._objectness_loss(predictions, targets))

    def compute_object_fabrication_gradient(self, image: np.ndarray) -> np.ndarray:
        """Compute the gradient that maximizes objectness for every prediction cell."""
        targets = self._targets_from_detections(None)
        for target in targets:
            target[..., 4] = 1.0
        return self._image_gradient(image, lambda predictions: self._objectness_loss(predictions, targets))

    def compute_object_untargeted_gradient(self, image: np.ndarray, detections: np.ndarray) -> np.ndarray:
        """Compute the negative YOLO loss gradient for benign detections."""
        targets = self._targets_from_detections(detections)
        return self._image_gradient(image, lambda predictions: -self._full_yolo_loss(predictions, targets))

    def compute_object_mislabeling_gradient(self, image: np.ndarray, detections: np.ndarray) -> np.ndarray:
        """Compute the YOLO loss gradient toward specified target detections."""
        targets = self._targets_from_detections(detections)
        return self._image_gradient(image, lambda predictions: self._full_yolo_loss(predictions, targets))


### TOG attack algorithms


In [ ]:
def _validate_attack_inputs(image: np.ndarray, num_iterations: int, epsilon: float, step_size: float) -> None:
    """Validate common attack parameters before running iterative updates."""
    if image.ndim != 4 or image.shape[-1] != 3:
        raise ValueError("image must have shape (batch, height, width, 3).")
    if num_iterations <= 0:
        raise ValueError("num_iterations must be positive.")
    if epsilon < 0 or step_size <= 0:
        raise ValueError("epsilon must be non-negative and step_size must be positive.")


def _initialize_adversarial_image(image: np.ndarray, epsilon: float) -> np.ndarray:
    """Randomly initialize an adversarial image inside the L-infinity constraint."""
    perturbation = np.random.uniform(-epsilon, epsilon, size=image.shape)
    return # TODO: clip the image + perturbation by 0.0 and 1.0. Use np.clip


def _project(image: np.ndarray, adversarial_image: np.ndarray, epsilon: float) -> np.ndarray:
    """Project an adversarial image into the valid pixel and L-infinity ranges."""
    perturbation = np.clip(adversarial_image - image, -epsilon, epsilon)
    return # TODO: clip the image + perturbation by 0.0 and 1.0. Use np.clip


def tog_vanishing(
    victim: YOLOv3TOGModel,
    image: np.ndarray,
    num_iterations: int = 10,
    epsilon: float = 8 / 255.0,
    step_size: float = 2 / 255.0,
) -> np.ndarray:
    """Generate a TOG object-vanishing adversarial image.

    Repeat the following for ``num_iterations``:
       - Compute the object-vanishing gradient.
       - Use the sign of the gradient to update the image in the direction
         that minimizes the vanishing objective.
       - Project the result back into the epsilon-constrained region and
         the valid pixel range ``[0, 1]``.
    """
    _validate_attack_inputs(image, num_iterations, epsilon, step_size)
    adversarial_image = _initialize_adversarial_image(image, epsilon)
    for _ in range(num_iterations):
        # TODO: update the adversarial image by calculating the object vanishing gradiant
    return adversarial_image


def tog_fabrication(
    victim: YOLOv3TOGModel,
    image: np.ndarray,
    num_iterations: int = 10,
    epsilon: float = 8 / 255.0,
    step_size: float = 2 / 255.0,
) -> np.ndarray:
    """Generate a TOG object-fabrication adversarial image.

    Repeat the following for ``num_iterations``:
       - Compute the object-fabrication gradient.
       - Update the image using the sign of the gradient in the direction
         that minimizes the fabrication objective.
       - Project the image back into the valid L-infinity and pixel ranges.

    The structure is similar to the vanishing attack, but it must call the
    fabrication-gradient method provided by ``victim``.
    """
    _validate_attack_inputs(image, num_iterations, epsilon, step_size)
    # create random noise for image
    adversarial_image = # TODO: initialize the random noisy image
    for _ in range(num_iterations):
        # TODO: student fill here
    return adversarial_image


def tog_mislabeling(
    victim: YOLOv3TOGModel,
    image: np.ndarray,
    target: Literal["most_likely", "least_likely"],
    num_iterations: int = 10,
    epsilon: float = 8 / 255.0,
    step_size: float = 2 / 255.0,
) -> np.ndarray:
    """Generate a targeted TOG object-mislabeling adversarial image.

    Your implementation should:

    1. Run the detector once on the original benign image.
    2. Convert the benign detections into targeted detections using
       ``generate_attack_targets()``.
       - ``"most_likely"`` selects a plausible alternative class.
       - ``"least_likely"`` selects the class with the lowest class score.
    3. Keep these target detections fixed throughout the attack.
    4. Randomly initialize the adversarial image inside the epsilon bound.
    5. For every iteration:
       - Compute the mislabeling gradient using the fixed target detections.
       - Apply a signed-gradient update.
       - Project the result into the valid perturbation and pixel ranges.

    Do not run target generation again inside the iterative loop.
    """
    # TODO: student fill here
    adversarial_image = _initialize_adversarial_image(image, epsilon)
    for _ in range(num_iterations):
        # TODO: student fill here
    return adversarial_image


def tog_untargeted(
    victim: YOLOv3TOGModel,
    image: np.ndarray,
    num_iterations: int = 10,
    epsilon: float = 8 / 255.0,
    step_size: float = 2 / 255.0,
) -> np.ndarray:
        """Generate an untargeted TOG adversarial image.

    Your implementation should:

    1. Run the detector once on the original benign image.
    2. Raise a clear ``ValueError`` if no benign objects are detected.
    3. Keep the original detections fixed during the iterative attack.
    4. Randomly initialize the adversarial image within the epsilon bound.
    5. For every iteration:
       - Compute the untargeted gradient using the fixed benign detections.
       - Apply the signed-gradient update expected by the provided attack
         model.
       - Project the result into the valid L-infinity and pixel ranges.

    The gradient method already defines the correct untargeted objective.
    """
    # TODO: student fill here
    adversarial_image = _initialize_adversarial_image(image, epsilon)
    for _ in range(num_iterations):
        # TODO: student fill here
    return adversarial_image

### Run code


Load the detector from local Darknet weights (`weights/yolov3.weights`).


In [ ]:
weights = ROOT / 'weights' / 'yolov3.weights'
detector = YOLOv3Detector(weights=weights)
victim = YOLOv3TOGModel(detector)
print('device:', detector.device)
print('num classes:', detector.num_classes)

Configuration of Attack Hyperparameters


In [ ]:
epsilon = 8 / 255.       # Maximum L-infinity perturbation
step_size = 2 / 255.     # Per-iteration update size
num_iterations = 10      # Number of attack iterations


Load and visualize the image to be attacked. Can choose different images


In [ ]:
image_path = ROOT / 'assets' / 'example_1.jpg'

input_img = Image.open(image_path).convert('RGB')
x_query, x_meta = letterbox_image(input_img, size=detector.model_image_size)
detections_query = detector.detect(x_query, confidence_threshold=detector.confidence_threshold)
visualize_detections({
    'Benign (No Attack)': (x_query, detections_query, detector.model_image_size, detector.class_names)
})
print('detections:', len(detections_query))


**TOG-untargeted Attack**

Random untargeted attacks fool the victim detector to randomly misdetect without targeting any specific object.


In [ ]:
# TODO: Run the TOG

"""
1. Call the tog_untargeted and pass the necessary params
2. Detect the adversarial image
3. Use visualize_detections to visualize
"""

**TOG-vanishing Attack**

TOG-vanishing aims at removing the victim's ability to identify objects (false negatives).


In [ ]:
# TODO: Run the TOG

"""
1. Call the tog_vanishing and pass the necessary params
2. Detect the adversarial image
3. Use visualize_detections to visualize
"""

**TOG-fabrication Attack**

TOG-fabrication fabricates additional detections (false positives).


In [ ]:
# TODO: Run the TOG

"""
1. Call the tog_fabrication and pass the necessary params
2. Detect the adversarial image
3. Use visualize_detections to visualize
"""


**TOG-mislabeling Attack**

Most-likely (ML) and least-likely (LL) class mislabeling attacks.


In [ ]:
# TODO: Run the TOG

"""
1. Call the tog_mislabeling and pass the necessary params, with 2 option most likely and least likely
2. Detect the adversarial image
3. Use visualize_detections to visualize
"""

## Questions and exploration

1. What happens if you increase or decrease the perturbation norm? Can you show some examples

2. How does the performance (mAP drop) change with different setting? What attack causes the most significant drop? Can you explain why

3. What happens if you change the signs of the gradients during TOG attacks?